# Baseline Model

## Table of Contents
1. [Model Choice](#model-choice)
2. [Feature Selection](#feature-selection)
3. [Implementation](#implementation)
4. [Evaluation](#evaluation)

In [ ]:
# Import necessary libraries
from pathlib import Path
import json

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid")

## Model Choice

The baseline model is a pretrained Microsoft ResNet-50 fine-tuned for 8-class SEM image classification. ResNet-50 was chosen because it is a well-established convolutional neural network for image classification and is suitable for transfer learning. Transfer learning is useful in this project because the SEM dataset is relatively small and imbalanced.

This baseline provides a reference point for future improvements. Later models can be compared against this result by adding more labeled data, tuning hyperparameters, trying other architectures, or combining image features with extracted microscope metadata.

## Feature Selection

The baseline model uses only the cropped SEM image pixels as input features. The white microscope information ribbon was removed before training so that the model focuses on the actual SEM structure.

Each image was resized to 224 x 224 pixels and converted to RGB format. Therefore, each sample has 224 x 224 x 3 = 150,528 pixel-level input features. The target variable is the image class label, with 8 possible categories: `3d_edge`, `Bond-Pad-Array`, `cantilever`, `close_up_line`, `Electrode`, `label`, `microfluidic`, and `waveguide`.

Microscope metadata was extracted separately from the white ribbon, but it was not used in this baseline model.

In [ ]:
# Load prepared dataset manifests
outputs_dir = Path(r"C:/Users/kom-e14-1/Documents/Codex/2026-06-05/i-have-sem-images-i-chategorized/outputs")
prepared_dir = outputs_dir / "prepared_sem_dataset"
model_dir = outputs_dir / "sem_resnet_model"

train_df = pd.read_csv(prepared_dir / "train_manifest.csv")
val_df = pd.read_csv(prepared_dir / "val_manifest.csv")

print(f"Training samples: {len(train_df)}")
print(f"Validation samples: {len(val_df)}")
print(f"Number of classes: {train_df['label'].nunique()}")
print(f"Input features per image: {224 * 224 * 3}")

display(train_df[["image_path", "label", "label_id", "is_augmented"]].head())

## Implementation

The baseline was implemented by fine-tuning `microsoft/resnet-50` using the Hugging Face Transformers library and PyTorch. The original ImageNet classification head was replaced with a new 8-class classification head. The model was trained on the prepared cropped SEM images.

Training configuration:
- Pretrained model: `microsoft/resnet-50`
- Number of classes: 8
- Epochs: 20
- Batch size: 16
- Learning rate: 0.00003
- Optimizer: AdamW
- Loss: weighted cross-entropy
- Backbone frozen for first 3 epochs
- Best checkpoint selected by validation accuracy

In [ ]:
# The model was trained with the project training script:
# python train_sem_resnet.py --data-dir ./prepared_sem_dataset --output-dir ./sem_resnet_model --model-name microsoft/resnet-50 --epochs 20 --batch-size 16 --learning-rate 0.00003

# Load saved training history and metrics
with open(model_dir / "history.json", "r", encoding="utf-8") as f:
    history = pd.DataFrame(json.load(f))

with open(model_dir / "metrics.json", "r", encoding="utf-8") as f:
    metrics = json.load(f)

display(history.tail())

## Evaluation

The baseline model was evaluated on the validation split using accuracy, precision, recall, and F1-score. Accuracy measures overall classification correctness. Precision and recall are important because the dataset is imbalanced. F1-score gives a balanced view of precision and recall and is useful for comparing performance across classes.

The baseline achieved 92.97% validation accuracy. The weighted average F1-score was approximately 0.927.

In [ ]:
# Summarize baseline performance
report = metrics["classification_report"]

print(f"Best validation accuracy: {metrics['best_val_accuracy']:.4f}")
print(f"Weighted F1-score: {report['weighted avg']['f1-score']:.4f}")
print(f"Macro F1-score: {report['macro avg']['f1-score']:.4f}")

class_names = metrics["class_names"]
class_rows = []
for class_name in class_names:
    row = report[class_name].copy()
    row["class_name"] = class_name
    class_rows.append(row)

class_report_df = pd.DataFrame(class_rows)[["class_name", "precision", "recall", "f1-score", "support"]]
display(class_report_df)

In [ ]:
# Plot validation accuracy over epochs
plt.figure(figsize=(8, 4))
plt.plot(history["epoch"], history["val_accuracy"], marker="o")
plt.title("Baseline ResNet-50 Validation Accuracy")
plt.xlabel("Epoch")
plt.ylabel("Validation accuracy")
plt.ylim(0, 1)
plt.tight_layout()
plt.show()

In [ ]:
# Confusion matrix
confusion = np.array(metrics["confusion_matrix"])

plt.figure(figsize=(8, 6))
sns.heatmap(confusion, annot=True, fmt="d", cmap="Blues", xticklabels=class_names, yticklabels=class_names)
plt.title("Baseline ResNet-50 Confusion Matrix")
plt.xlabel("Predicted label")
plt.ylabel("True label")
plt.xticks(rotation=45, ha="right")
plt.yticks(rotation=0)
plt.tight_layout()
plt.show()

### Baseline Interpretation

The baseline model performs well overall, with strong validation performance for classes such as `Bond-Pad-Array`, `3d_edge`, `waveguide`, and `cantilever`. The main confusion occurs between `close_up_line` and `Electrode`. The minority classes `label` and `microfluidic` are less reliable because they have very few real validation examples.

This baseline is a useful reference for future model improvements.